# DiGToR — Demo Streamlit truc tiep tren Google Colab

Tai len **1 anh RGB + 1 anh nhiet** -> mo hinh DiGToR chay that -> hien **segmentation map**
va **routing map** (giong `digtor_figures_and_gaps.ipynb`).

**Cach dung:** Runtime -> Change runtime type -> **GPU**, roi Run-All cac o duoi.
O cuoi se in ra mot duong link `https://...trycloudflare.com` — bam vao do la thay giao dien.

## 1. Clone repo + cai dat

In [ ]:
import os
REPO_URL = "https://github.com/nguyenmaiductrong/DiGToR.git"
if not os.path.isdir("DiGToR"):
    os.system(f"git clone {REPO_URL}")
os.system("cd DiGToR && git pull origin main")
%cd /content/DiGToR
!pip -q install -r demo/requirements.txt
!pip -q install torch numpy pillow scikit-learn matplotlib gdown wandb streamlit
print("OK")

## 2. Lay checkpoint `digtor.pt`

Chon **MOT** trong hai cach duoi (W&B hoac Google Drive). Sau buoc nay phai co
`ckpt_fmb/digtor.pt` (hoac `ckpt_semanticrt/digtor.pt`).

### 2a. (Cach A) Keo tu Weights & Biases

In [ ]:
DATASET       = "fmb"            # "fmb" hoac "semanticrt"
WANDB_PROJECT = "digtor-fmb"     # dat None de bo qua W&B
WANDB_ENTITY  = None             # username/entity cua ban, hoac None
WANDB_ALIAS   = "latest"
CKPT_DIR      = "ckpt_fmb"       # noi ckpt se nam

import os
if WANDB_PROJECT:
    os.system("pip -q install wandb")
    # can WANDB_API_KEY trong moi truong, hoac chay: import wandb; wandb.login()
    from digtor.wandb_ckpt import pull_checkpoint
    ok = pull_checkpoint("digtor", CKPT_DIR, WANDB_PROJECT, WANDB_ENTITY, WANDB_ALIAS)
    print("digtor.pt:", ok)
print("ton tai:", os.path.exists(f"{CKPT_DIR}/digtor.pt"))

### 2b. (Cach B) Mount Google Drive roi copy ckpt

In [ ]:
# from google.colab import drive; drive.mount("/content/drive")
# !mkdir -p ckpt_fmb
# !cp "/content/drive/MyDrive/<thu_muc_ckpt>/digtor.pt" ckpt_fmb/digtor.pt
# import os; print(os.path.exists("ckpt_fmb/digtor.pt"))

## 3. Khoi dong Streamlit + mo tunnel cong khai

Chay o duoi roi cho ~15-20 giay. No se in **mot URL `trycloudflare.com`** — bam vao do.
Trong giao dien: kiem tra thanh ben (duong dan `digtor.pt`, chon fmb/semanticrt), roi
tai len anh RGB + anh nhiet de xem segmentation map va routing map.

In [ ]:
import subprocess, time, re, os, urllib.request

# tai cloudflared (tunnel khong can dang ky)
if not os.path.exists("cloudflared"):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "cloudflared")
    os.chmod("cloudflared", 0o755)

# chay streamlit nen
os.system("pkill -f streamlit; pkill -f cloudflared; sleep 1")
st_log = open("streamlit.log", "w")
subprocess.Popen(
    ["streamlit", "run", "demo/app_live.py",
     "--server.port", "8501", "--server.headless", "true",
     "--server.address", "0.0.0.0", "--browser.gatherUsageStats", "false"],
    stdout=st_log, stderr=st_log)
time.sleep(8)

# mo tunnel
tun = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(40):
    line = tun.stdout.readline()
    if not line:
        time.sleep(0.5); continue
    m = re.search(r"https://[-\w.]+trycloudflare\.com", line)
    if m:
        url = m.group(0); break
print("\n" + "=" * 60)
print(" MO GIAO DIEN STREAMLIT TAI:" )
print(" ", url or "(khong lay duoc URL — xem streamlit.log / output tren)")
print("=" * 60)